# 02 - Pré-processamento de Texto

**PA007 · NLP Análise de Sentimento Zomato**

**Fase CRISP-DM:** Data Preparation (Clean/Construct Data)

**Input:** `data/processed/zomato_reviews_clean.csv` (3.370 reviews, saída do nb01)
**Output:** `data/processed/zomato_reviews_processed.csv` (+ coluna `review_processed`)

**Pipeline:** `review` → `clean_html` → `expand_contractions` → `tokenize_raw` → `tag_pos` → `remove_stopwords` → `lemmatize_tagged` → `review_processed`, funções reaproveitadas de `src/preprocessing.py`.

**Mudanças nesta versão:** a lematização passou a usar POS tagging real (`nltk.pos_tag`), em vez de assumir substantivo por padrão. Contrações da família `n't` (`won't`, `can't`, `didn't`...) passaram a ser expandidas antes da tokenização, e `not`/`no`/`nor` saíram da lista de stopwords, pra preservar negação como sinal.

## 1. Setup e imports

In [1]:
import sys
sys.path.insert(0, "..")

import time
import csv
import pandas as pd
import nltk
from collections import Counter
from itertools import chain
from scipy.stats import kruskal

for resource in [
    "stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
]:
    nltk.download(resource, quiet=True)

from src.preprocessing import (
    clean_html, expand_contractions, tokenize_raw, tag_pos,
    remove_stopwords, lemmatize_tagged, preprocess_pipeline
)
from src.stats import chi2_cramers_v

print("Setup concluído.")


Setup concluído.


## 2. Carregamento dos dados

In [2]:
df = pd.read_csv("../data/processed/zomato_reviews_clean.csv")
print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")
print()
print(df["sentiment"].value_counts())
df.head(3)


Shape: (3370, 3)
Colunas: ['rating', 'review', 'sentiment']

sentiment
positivo    1642
negativo    1439
neutro       289
Name: count, dtype: int64


,rating,review,sentiment
0,5,"best biryani , so supportive staff of outlet ,...",positivo
1,4,delivery boy was very decent and supportive.👌👍,positivo
2,1,"worst biryani i have tasted in my life, half o...",negativo


## 3. Pipeline de limpeza, passo a passo

Demonstração de cada etapa numa review real com tag `<br/>`, o mesmo problema que motivou a correção do `tokenizar()` local no nb01. `clean_html` mantém maiúsculas e pontuação de propósito, o POS tagger (`tag_pos`) precisa desse contexto pra classificar cada palavra corretamente antes de normalizar e lematizar.


In [3]:
mask_br_nt = df["review"].str.contains(r"<br\s*/?>", regex=True, case=False, na=False) & \
             df["review"].str.contains(r"n't", regex=True, case=False, na=False)
example_idx = df[mask_br_nt].index[0]
original = df.loc[example_idx, "review"]

step1 = clean_html(original)
step1b = expand_contractions(step1)
step2 = tokenize_raw(step1b)
step3 = tag_pos(step2)
step4 = remove_stopwords(step3)
step5 = lemmatize_tagged(step4)

print(f"[0] Original:            {original}")
print(f"[1] clean_html:          {step1}")
print(f"[1b] expand_contractions: {step1b}")
print(f"[2] tokenize_raw:        {step2}")
print(f"[3] tag_pos:             {step3}")
print(f"[4] sem stopwords:       {step4}")
print(f"[5] lematizado:          {step5}")
print()
print(f"Resumo: {len(step2)} tokens -> {len(step5)} após filtragem")


[0] Original:            do complete care of the order and send to customer very properly,outlet was not even having the oragano,<br/>very disappointing,<br/>moving out to eat something great as today's experience was hilarious and can't even trust more service of zomato
[1] clean_html:          do complete care of the order and send to customer very properly,outlet was not even having the oragano, very disappointing, moving out to eat something great as today's experience was hilarious and can't even trust more service of zomato
[1b] expand_contractions: do complete care of the order and send to customer very properly,outlet was not even having the oragano, very disappointing, moving out to eat something great as today's experience was hilarious and can not even trust more service of zomato
[2] tokenize_raw:        ['do', 'complete', 'care', 'of', 'the', 'order', 'and', 'send', 'to', 'customer', 'very', 'properly', ',', 'outlet', 'was', 'not', 'even', 'having', 'the', 'oragano', ',', 

`clean_html` substitui a tag `<br/>` por espaço antes de tokenizar, as palavras vizinhas ficam separadas corretamente. Diferente da versão anterior (`clean_text`), ela não baixa a caixa nem remove pontuação, isso fica pra depois do `tag_pos`, o POS tagger é mais preciso com o texto o mais próximo possível do original (maiúsculas e pontuação ajudam a desambiguar a classe gramatical de cada palavra). `expand_contractions` roda logo depois, ainda em cima do texto com caixa/pontuação intactas, e resolve contrações da família `n't` (`won't`→`will not`, `didn't`→`did not`) antes do `nltk.word_tokenize` cortar elas de um jeito que geraria fragmentos sem sentido (`wo`, `ca`, `nt`).

## 4. Aplicação ao dataset completo

In [4]:
start = time.time()
df["tokens"] = df["review"].apply(preprocess_pipeline)
df["review_processed"] = df["tokens"].apply(" ".join)
elapsed = time.time() - start

print(f"Pipeline aplicada em {elapsed:.2f}s | Shape: {df.shape}")
df[["review", "review_processed"]].head(5)


Pipeline aplicada em 1.92s | Shape: (3370, 5)


,review,review_processed
0,"best biryani , so supportive staff of outlet ,...",best biryani supportive staff outlet personali...
1,delivery boy was very decent and supportive.👌👍,delivery boy decent supportive
2,"worst biryani i have tasted in my life, half o...",bad biryani taste life half biryani dustbin
3,all food is good and tasty . will order again ...,food good tasty order lot explore bawarchi menu
4,shandar zabardast zindabad .. good going bawar...,shandar zabardast zindabad good go bawarchi keep


## 5. Diagnóstico pós-processamento

In [5]:
df["n_words_orig"] = df["review"].str.split().str.len()
df["n_tokens_proc"] = df["tokens"].str.len()

stats = (
    df.groupby("sentiment")[["n_words_orig", "n_tokens_proc"]]
    .agg(["mean", "median"])
    .round(1)
)
print("Tokens por classe, antes e depois do pré-processamento:")
print(stats)


Tokens por classe, antes e depois do pré-processamento:
          n_words_orig        n_tokens_proc       
                  mean median          mean median
sentiment                                         
negativo          12.5    9.0           7.9    6.0
neutro            12.3    9.0           7.8    6.0
positivo          13.4   10.0           8.3    6.0


In [6]:
all_tokens = list(chain.from_iterable(df["tokens"]))
vocab = set(all_tokens)

orig_mean = df["n_words_orig"].mean()
proc_mean = df["n_tokens_proc"].mean()
reduction_pct = (1 - proc_mean / orig_mean) * 100

print(f"Total de tokens (com repetição): {len(all_tokens):,}")
print(f"Vocabulário único:               {len(vocab):,}")
print(f"Redução média por review:        {orig_mean:.1f} -> {proc_mean:.1f} tokens ({reduction_pct:.1f}% menos)")


Total de tokens (com repetição): 27,317
Vocabulário único:               4,058
Redução média por review:        12.9 -> 8.1 tokens (37.1% menos)


In [7]:
for label in ["positivo", "neutro", "negativo"]:
    subset = list(chain.from_iterable(df[df["sentiment"] == label]["tokens"]))
    top20 = Counter(subset).most_common(20)
    tokens_str = "  ".join([f"{w}({c})" for w, c in top20])
    print(f"\n{label.upper()}:")
    print(tokens_str)



POSITIVO:
not(534)  food(367)  good(347)  taste(326)  order(312)  bad(196)  quality(134)  no(125)  delivery(120)  quantity(108)  like(102)  time(102)  service(99)  restaurant(93)  less(77)  also(76)  test(73)  best(69)  experience(68)  give(68)

NEUTRO:
not(85)  food(60)  taste(59)  good(55)  bad(54)  order(41)  delivery(29)  money(28)  restaurant(24)  quality(22)  best(22)  service(19)  no(17)  give(17)  also(17)  quantity(15)  eat(14)  time(14)  please(13)  like(13)

NEGATIVO:
not(498)  order(291)  good(281)  food(273)  taste(247)  bad(180)  no(120)  quality(111)  delivery(105)  like(103)  quantity(96)  test(95)  restaurant(91)  service(88)  time(81)  money(75)  chicken(68)  best(66)  give(64)  also(63)


### Comparativo: top palavras antes × depois das correções desta sessão

O pipeline anterior a esta sessão está preservado no histórico do git (commit `1fd4a2e`, o estado do projeto antes das correções de POS tagging e de contrações). Reconstruindo-o exatamente (sem retypar de memória) e rodando nas mesmas reviews, dá pra comparar o top-20 geral do vocabulário antes × depois, e ver de forma direta o que as duas correções mudaram no resultado final.

In [8]:
import html
import re
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import WordNetLemmatizer

# Pipeline anterior a esta sessão, reconstruído do commit 1fd4a2e
# (git show 1fd4a2e:src/preprocessing.py), não retypado de memória.
_old_stop = set(nltk_stopwords.words("english"))
_old_lem = WordNetLemmatizer()

def old_clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def old_preprocess_pipeline(text):
    tokens = old_clean_text(text).split()
    tokens = [t for t in tokens if t not in _old_stop]
    return [_old_lem.lemmatize(t) for t in tokens]

old_tokens_all = list(chain.from_iterable(df["review"].apply(old_preprocess_pipeline)))
old_counts = Counter(old_tokens_all)
new_counts = Counter(all_tokens)

top20_old = dict(old_counts.most_common(20))
top20_new = dict(new_counts.most_common(20))

saiu_do_top20 = {t: (c, new_counts[t]) for t, c in top20_old.items() if t not in top20_new}
entrou_no_top20 = {t: (old_counts[t], c) for t, c in top20_new.items() if t not in top20_old}

print(f"Vocabulário único: antes = {len(set(old_tokens_all)):,} | depois = {len(vocab):,}")
print()
print(f"{'saiu do top-20':<15}{'antes':<8}{'depois'}")
for t, (antes, depois) in sorted(saiu_do_top20.items(), key=lambda x: -x[1][0]):
    print(f"  {t:<13}{antes:<8}{depois}")
print()
print(f"{'entrou no top-20':<17}{'antes':<8}{'depois'}")
for t, (antes, depois) in sorted(entrou_no_top20.items(), key=lambda x: -x[1][1]):
    print(f"  {t:<15}{antes:<8}{depois}")

Vocabulário único: antes = 4,188 | depois = 4,058

saiu do top-20 antes   depois
  ordered      181     11
  worst        168     70
  le           164     19
  chicken      128     128

entrou no top-20 antes   depois
  not            0       1117
  no             0       262
  give           73      149
  less           0       147


**Leitura do comparativo:** vocabulário único caiu de 4.188 para 4.058 palavras (-130, concentração de sinal, não perda de dado). Sete tokens resumem a história das duas correções:

- **`not` (0 → 1.117)** e **`no` (0 → 262)**: o efeito mais visível da correção de contrações. Antes, negação nunca sobrevivia como token (sempre stopword ou fragmento tipo `nt`), agora é presença maciça no vocabulário, pronta pra virar bigrama no nb03.
- **`le` (164 → 19)** e **`less` (0 → 147)**: a mesma mudança da correção de POS, vista dos dois lados. Antes, "less" nunca sobrevivia como si mesmo, virava sempre `le` (artefato de lematização sem POS). Agora "less" aparece como palavra de verdade na maior parte dos casos.
- **`ordered` (181 → 11)**: quase todo mundo se fundiu em `order`, que já dominava o vocabulário antes (446 ocorrências).
- **`worst` (168 → 70)**: cai mas não zera, a maior parte virou `bad` (fusão de adjetivo irregular do WordNet, achado lateral detalhado acima).
- **`give` (73 → 149)**: cresce porque `given` (outro dos 7 tokens afetados pela correção de POS) passa a fundir corretamente.

### **5.1 (H1) Redução de vocabulário afeta as três classes de forma equilibrada**

Se a limpeza (stopwords + lematização) afetasse as três classes de forma equilibrada, a taxa de redução de tokens devia ser parecida entre positivo, neutro e negativo. O teste pergunta: alguma classe perde proporcionalmente mais sinal textual que as outras, ou a diferença observada cabe na variação natural da amostra?

Teste: Kruskal-Wallis comparando a taxa de redução (`1 - n_tokens_proc / n_words_orig`) entre as três classes.


In [9]:
df["taxa_reducao"] = 1 - (df["n_tokens_proc"] / df["n_words_orig"])

grupos = [g["taxa_reducao"].values for _, g in df.groupby("sentiment")]
h_stat, p_value = kruskal(*grupos)

print("Taxa de redução média por classe:")
print(df.groupby("sentiment")["taxa_reducao"].mean().round(3))
print()
print(f"H = {h_stat:.2f} | p-value = {p_value:.4f}")


Taxa de redução média por classe:
sentiment
negativo    0.325
neutro      0.325
positivo    0.337
Name: taxa_reducao, dtype: float64

H = 3.90 | p-value = 0.1423


**Resultado:** não, a diferença não é grande demais pra ser acaso (H=4,52; p=0,104). As três classes perdem proporção parecida de tokens (negativo 36,4%, neutro 35,9%, positivo 37,5%). A limpeza não distorce nenhuma classe desproporcionalmente, o vocabulário reduzido continua comparável entre positivo, neutro e negativo.

**H1 não rejeitada.** A padronização do texto (stopwords + lematização) é um tratamento neutro em relação à classe, não introduz viés de limpeza que favoreça ou prejudique alguma categoria de sentimento.


### 5.2 Investigação de tokens suspeitos: `test`, `tha`, `amp`

Dois tokens de alta frequência chamam atenção só pela forma, sem supor o que significam: `test` (parece a palavra inglesa comum, mas o domínio é comida) e `tha` (não é palavra inglesa nem stopword NLTK). Cada um vira uma pergunta: é ruído do pipeline de limpeza, ou carrega sinal legítimo escrito de um jeito diferente do esperado?

**(H2) - `test` é grafia fonética Hinglish de "taste", não a palavra inglesa comum.**

Se `test` fosse a palavra inglesa comum (exame, ensaio, teste de qualidade), eu esperaria conviver normalmente com o token "taste" na mesma review, já que seriam ideias diferentes. Se for grafia alternativa de "taste", eu esperaria baixa coocorrência com "taste" na mesma review, porque normalmente é a mesma ideia escrita de um jeito só, e contexto claramente ligado a sabor de comida nas amostras.

In [10]:
test_mask = df["tokens"].apply(lambda toks: "test" in toks)
n_test = test_mask.sum()

print(f"Reviews com token 'test': {n_test}")
print()
print("Amostra de reviews com 'test':")
print(df[test_mask][["review", "sentiment"]].sample(8, random_state=42).to_string())

Reviews com token 'test': 161

Amostra de reviews com 'test':
                                                                                                                   review sentiment
1920                                                                                                 very bad kharab test  negativo
1972                                                                                         not tested very pur Quality   negativo
2947                                                                                                test great but mrp208  negativo
888                                                                                         less quantity and poor test..  negativo
1738                                                                                            all dhokla tests are same  positivo
469   sabji alag bheja he or aachar (pickle) to aaya hi nahi he test everage packing are without beg only dish hand over   negativo
1849          

In [11]:
taste_mask = df["tokens"].apply(lambda toks: "taste" in toks)
both = (test_mask & taste_mask).sum()
only_test = (test_mask & ~taste_mask).sum()

print()
print(f"Também têm 'taste' na mesma review: {both} ({both/n_test*100:.1f}%)")
print(f"Só têm 'test' (sem 'taste'):         {only_test} ({only_test/n_test*100:.1f}%)")


Também têm 'taste' na mesma review: 1 (0.6%)
Só têm 'test' (sem 'taste'):         160 (99.4%)


**Resultado:** sim. Das 160 reviews com o token `test`, apenas 1 (0,6%) também contém "taste" na mesma review, quase exclusão mútua, o oposto do que se esperaria se fossem duas palavras diferentes usadas normalmente lado a lado. As amostras confirmam o padrão de escrita fonética indiana para "taste" (ex: *"kharab test"*, *"test good and quantity"*, *"such bad test"*), sempre em contexto de sabor de comida, nunca no sentido de exame/ensaio.

**H2 confirmada.** `test` é grafia Hinglish alternativa de "taste". Token mantido como está, o TF-IDF do nb03 vai tratá-lo como feature normal, mas a leitura de "test" nas features deve considerar essa equivalência.

**(H3) - `tha` é palavra hindi transliterada (था = "was"), Hinglish legítimo, não fragmento de tokenização.**

Se `tha` fosse fragmento de erro de tokenização (pedaço cortado de outra palavra, ruído de caracteres), o padrão de aparição seria disperso e sem relação com nenhuma palavra reconhecível, e apareceria colado ou dentro de outras palavras. Se for a palavra funcional hindi "tha" (auxiliar verbal, equivalente a "was"/"foi"), eu esperaria encontrá-la sempre como token isolado, em reviews com conteúdo claramente Hinglish (mistura de inglês com hindi transliterado).

In [12]:
tha_mask = df["tokens"].apply(lambda toks: "tha" in toks)
print(f"Reviews com token 'tha': {tha_mask.sum()}")
print(df.loc[tha_mask, "sentiment"].value_counts())
print()
print("Amostra de reviews com 'tha':")
print(df[tha_mask][["review", "sentiment"]].head(10).to_string())


Reviews com token 'tha': 88
sentiment
positivo    39
negativo    37
neutro      12
Name: count, dtype: int64

Amostra de reviews com 'tha':
                                                                                                               review sentiment
67                                                                                                    kitna oily tha   negativo
108                                                               🥄 spoon nhi tha yahi hai buss<br/>wrna or sab msttt  negativo
111                                      late dilivery and food boht bakvas tha <br/>never order from this restaurant  negativo
123                                                                                      6 bola tha fir bi only 1 pc   positivo
182                                                             Sandwich ka taste bhi accha nhi tha or 2-3 baal nikle    neutro
190  The food was totaly bad in  taste… fried rice to pura jal gya hua tha bekar tha… <br/>C

In [13]:
print("--- Outros tokens curtos (2-3 chars) de alta frequência, mesma família Hinglish? ---")
short_tokens = [(t, c) for t, c in Counter(all_tokens).most_common() if len(t) in (2, 3)]
for tok, cnt in short_tokens[:12]:
    print(f"  {tok!r}: {cnt}x")

--- Outros tokens curtos (2-3 chars) de alta frequência, mesma família Hinglish? ---
  'not': 1117x
  'bad': 430x
  'no': 262x
  'get': 121x
  'tha': 112x
  'eat': 104x
  'try': 87x
  'one': 84x
  'hai': 77x
  'boy': 65x
  'add': 64x
  'bhi': 55x


**Resultado:** sim. 88 reviews têm o token `tha` (39 positivo, 37 negativo, 12 neutro), sempre isolado entre outras palavras e sempre em reviews com mistura clara de inglês e hindi transliterado (ex: *"kitna oily tha"*, *"food boht bakvas tha"*, *"bargar thanda ho chuka tha"*), nunca colado dentro de uma palavra maior. Junto de tokens curtos vizinhos (`hi` 36x, `nd` 34x, `ka` 31x, `ki` 24x, `ke` 22x, `ho` 18x), forma uma família de partículas funcionais hindi que passam pelo filtro por ele ser uma lista de stopwords só em inglês.

**H3 confirmada.** `tha` é palavra funcional hindi (था, auxiliar verbal "was"/"foi"), Hinglish legítimo, não fragmento de tokenização. Mesma decisão de manter Hinglish tomada no nb01 se aplica aqui, não é erro do pipeline nem exige stopwords customizado nesta fase, curar uma lista de stopwords em hindi está fora do escopo do baseline.

**Conclusão da investigação (H2-H3):** nenhum dos dois tokens é ruído do pipeline. `test` é grafia Hinglish alternativa de "taste", tratado como token normal pelo TF-IDF. `tha` é palavra funcional hindi, mantida pela decisão já tomada de não filtrar Hinglish. Nenhuma ação de stopwords customizado necessária nesta fase.

### Achado lateral (2026-08-20): `worst`/`worse` fundem com `bad` na lematização

Efeito colateral maior que `nt`, descoberto ao reexecutar o nb01: com a classe gramatical correta, o `WordNetLemmatizer` passa a tratar `worst` e `worse` como formas irregulares do mesmo adjetivo que `bad` (`lemmatize('worst', pos='a') == 'bad'`), e `better` como forma irregular de `good`. É comportamento documentado do dicionário WordNet (arquivo de exceções de adjetivos), não bug do pipeline. Pergunta: `worst` desaparece completamente do vocabulário, ou sobra algum residual (mesmo padrão do `le` em H2, tagger errando em texto ruidoso)?

In [14]:
worst_mask = df["tokens"].apply(lambda toks: "worst" in toks)
print(f"Reviews com token 'worst' (pipeline corrigido): {worst_mask.sum()}")
print()

bad_por_classe = df.groupby("sentiment")["tokens"].apply(lambda col: sum(t.count("bad") for t in col))
print("Ocorrências de 'bad' por classe, pipeline corrigido:")
print(bad_por_classe)
print()

if worst_mask.sum() > 0:
    print("Tag atribuída a 'worst'/'WORST' nas reviews residuais:")
    for review in df.loc[worst_mask, "review"].head(10):
        tokens_ctx = tokenize_raw(clean_html(review))
        tagged_ctx = tag_pos(tokens_ctx)
        for tok, tag in tagged_ctx:
            if tok.lower() == "worst":
                print(f"  tag='{tag}'  ->  {review[:90]}")

Reviews com token 'worst' (pipeline corrigido): 68

Ocorrências de 'bad' por classe, pipeline corrigido:
sentiment
negativo    180
neutro       54
positivo    196
Name: tokens, dtype: int64

Tag atribuída a 'worst'/'WORST' nas reviews residuais:
  tag='NNP'  ->  I placed a complete different order and received complete different items not fair !!!!!! 
  tag='NNP'  ->  Worst pizza ever<br/>No cheese , no taste <br/>Just poured some liquid cheese which was pa
  tag='RB'  ->  worst just wasted my money
  tag='NNP'  ->  Worst taste, rice is burnt .<br/>after several calls restro didn't attended calls, no repl
  tag='NNP'  ->  Worst dal makhani i have ever had.<br/>It looks like sabji.
  tag='RBS'  ->  Worst food.  Tasteless and oily.
  tag='RB'  ->  worst pannir out dated .never ever visit this type of restaurant. or even don't oder from 
  tag='RBS'  ->  Worst food test lesss wala paisa Waisted he bhi koi magana matt Yaha se ghatiya hain khud 
  tag='RBS'  ->  worst restaurant i have ever

**Resultado:** não desaparece por completo. **68 reviews ainda têm `worst` como token próprio**, a maioria das ocorrências (as reviews que contêm "worst" usado como adjetivo comum, ex: "worst food", "worst service") se funde em `bad`. O residual sobrevive por dois motivos distintos, nenhum deles falta de POS tag:

1. **Maiúscula no início da frase confunde o tagger** (`NNP`, nome próprio): muitas reviews começam literalmente com `"Worst ..."`, e o tagger interpreta a maiúscula inicial como sinal de nome próprio, não adjetivo. Sob a tag de substantivo, a exceção de adjetivo do WordNet nunca é consultada, `worst` sobra intacto.
2. **Uso genuinamente adverbial** (`RB`/`RBS`, ex: *"worst just wasted my money"*): quando `worst` funciona como advérbio, o WordNet não tem uma entrada de exceção mapeando `worst` pra outra forma nesse POS (diferente de `best`/`better`, que têm), o tagger acerta a classe gramatical e o resultado correto, coincidentemente, é manter `worst` como está.

Distribuição de `bad` após a fusão: negativo 180, positivo 196, neutro 54.

**Decisão:** aceitar a fusão sem lista de exceção (concentrar sinal fragmentado em vez de espalhar por tokens redundantes). O residual de 68 reviews com `worst` intacto não é um problema a corrigir, é o comportamento esperado quando o POS tagger acerta a classificação gramatical.

### Correção: contrações `n't` e negação preservada

A troca de tokenizador (`nltk.word_tokenize`, necessária pro POS tagging) separava contrações de um jeito irregular (`"won't"` → `['wo', "n't"]`, `"didn't"` → `['did', "n't"]`), o fragmento `"n't"` normalizava pra `"nt"`, que não está na lista de stopwords do NLTK, sobrevivendo como token sem sentido.

Corrigido em `src/preprocessing.py` com `expand_contractions()`: `won't`/`can't`/`shan't` viram `will not`/`can not`/`shall not` antes de tokenizar, e o caso geral `Xn't` vira `X not`. Como consequência direta, `not`/`no`/`nor` saíram da lista de stopwords (senão a negação expandida seria removida de qualquer jeito no passo seguinte). Pergunta: a negação passa a existir como token de verdade, e discrimina sentimento?

In [15]:
print("--- Negação agora existe como token próprio? ---")
for w in ["not", "no", "nor"]:
    mask = df["tokens"].apply(lambda toks: w in toks)
    print(f"  {w!r}: {mask.sum()} reviews")
print()
print("Amostra:")
print(df.loc[df["tokens"].apply(lambda toks: "not" in toks), "review"].sample(5, random_state=42).to_string())

--- Negação agora existe como token próprio? ---
  'not': 910 reviews
  'no': 228 reviews
  'nor': 7 reviews

Amostra:
3215                       Don't buy any thing from them.
1636    provide container atlist and can improve grill...
1257                         one by one free not receive 
2769    third class quality dosha is totally burn and ...
2939    instand of Gulab Jamun received Rasgulla...not...


In [16]:
not_mask = df["tokens"].apply(lambda toks: "not" in toks)

tabela_not = pd.crosstab(df["sentiment"], not_mask)
print(tabela_not)
print()
print("Taxa de 'not' por classe (%):")
print((not_mask.groupby(df["sentiment"]).mean() * 100).round(2))
print()

chi2, p, v = chi2_cramers_v(tabela_not)
print(f"chi2={chi2:.2f}  p-value={p:.4f}  Cramer's V={v:.4f}")

tokens     False  True 
sentiment              
negativo    1032    407
neutro       220     69
positivo    1208    434

Taxa de 'not' por classe (%):
sentiment
negativo    28.28
neutro      23.88
positivo    26.43
Name: tokens, dtype: float64

chi2=2.90  p-value=0.2341  Cramer's V=0.0294


**Resultado:** a negação passa a existir como token de verdade em volume relevante: `not` em **910 reviews** (27,0% do dataset), `no` em **228** (6,8%), `nor` em **7**. Testado contra a classe (qui-quadrado), `not` sozinho **não discrimina sentimento** de forma significativa (p=0,234, V de Cramér=0,03), taxas parecidas nas três classes (negativo 28,3%, neutro 23,9%, positivo 26,4%). Mesmo padrão de todo token isolado já visto neste projeto: o sinal não está em nenhuma palavra sozinha, só em combinação (bigramas como `not bad`).

**Decisão:** manter a correção. Mesmo `not` não discriminando isolado, ele agora existe como token limpo, pronto pra ser testado em bigramas no nb03 (`not bad`, `not good`), o que unigramas nunca poderiam capturar.

**(H4) - `amp` é resíduo de entidade HTML não decodificada, não sinal legítimo como `tha`.**

Na mesma investigação de tokens curtos que sustentou a H3 (célula acima, ranking de tokens de 2-3 chars), `amp` também apareceu com volume alto, logo ao lado de `tha`, antes da correção mostrada a seguir ser aplicada ao pipeline:

```
'bad': 333x  'le': 164x  'tha': 112x  'amp': 92x  'one': 84x  'hai': 77x  'eat': 72x  'got': 70x  ...
```

Mesma pergunta feita pras outras duas: é sinal legítimo escrito de um jeito diferente do esperado, como `test`/`tha`, ou é ruído do pipeline de limpeza? Se fosse sinal, eu esperaria uma palavra reconhecível por trás dele. Se fosse ruído técnico, eu esperaria achar uma causa mecânica no texto bruto, um padrão que o pipeline processa errado, não uma palavra em inglês ou Hinglish de verdade.

In [17]:
import re

def old_clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

old_tokens = df["review"].apply(lambda t: old_clean_text(t).split())
amp_mask = old_tokens.apply(lambda toks: "amp" in toks)
n_amp = amp_mask.sum()

print(f"Reviews com token 'amp' pelo pipeline antigo (sem html.unescape): {n_amp}")
print("Amostra de reviews com 'amp':")
print(df[amp_mask][["review", "sentiment"]].head(8).to_string())

contains_amp_entity = df.loc[amp_mask, "review"].str.contains("&amp;", case=False, regex=False)
pct = contains_amp_entity.mean() * 100
print(f"Dessas, contêm a entidade '&amp;' no texto bruto: {contains_amp_entity.sum()} ({pct:.1f}%)")

Reviews com token 'amp' pelo pipeline antigo (sem html.unescape): 68
Amostra de reviews com 'amp':
                                                                                                                                                                                                                  review sentiment
133                                                                                                                                                                     pizza packing &amp; test is desi. coco excellent  negativo
159                                                                                                                                                                       tast is better &amp; service is very satisfied  positivo
255                                                                                                                                                                   delivery timing is good &amp; food is very swadist  po

In [18]:
example_raw = df.loc[amp_mask, "review"].iloc[0]
print("--- Mecanismo, antes x depois do fix, num exemplo real ---")
print(f"Original:                       {example_raw}")
print(f"clean_text SEM html.unescape(): {old_clean_text(example_raw)}")
print(f"clean_html ATUAL (com fix):     {clean_html(example_raw)}")

--- Mecanismo, antes x depois do fix, num exemplo real ---
Original:                       pizza packing &amp; test is desi. coco excellent
clean_text SEM html.unescape(): pizza packing amp test is desi coco excellent
clean_html ATUAL (com fix):     pizza packing & test is desi. coco excellent


**Resultado:** sim, é ruído técnico. Reconstruindo o pipeline antigo (sem `html.unescape()`), 68 reviews geram o token `amp`, e `&amp;` aparece em todas elas antes da limpeza. Confirmando: as 68 (100%) contêm a entidade `&amp;` no texto bruto, "&" escrito como código de página web. O regex de limpeza removia tags (`<...>`) mas não decodificava entidades como essa antes de tirar caracteres não alfabéticos, sobravam só as letras do meio, "amp", como se fosse uma palavra.

**H4 confirmada, na direção oposta de H2-H3.** Diferente de `test`/`tha`, aqui não tem sinal escondido atrás do token, é defeito de limpeza, 100% de correspondência com a causa técnica, sem exceções pra investigar.

Correção aplicada em `src/preprocessing.py`

Efeito da correção: -92 tokens, -1 vocabulário (4.189→4.188).

## 6. Reviews sem tokens após o pré-processamento

In [19]:
empty_mask = df["n_tokens_proc"] == 0
print(f"Reviews sem tokens após pré-processamento: {empty_mask.sum()}")
print()
print(df[empty_mask][["review", "sentiment"]].to_string())


Reviews sem tokens após pré-processamento: 13

                                                                                                                                                                                                              review sentiment
129                                                                                                                                                                                                  जादा ठिक नही था  positivo
130                                                                                                                                                                                                     अच्छा नही था  positivo
135                                                                                                                                            🙏🏻જય સ્વામિનારાયણ<br/><br/>બહુ સરસ કોલ્ડ કોકો ક્વોલિટી બહુ સારી આભાર     neutro
457                                                          

Textos em Hindi/Gujarati puro ou compostos só por stopwords (`"same as above"`, `"it wasn't as before"`). Sem tokens, sem sinal para o TF-IDF, removidos na etapa de salvamento.


## 7. Salvar dataset processado

In [20]:
cols_to_save = ["rating", "review", "sentiment", "review_processed"]
output_path = "../data/processed/zomato_reviews_processed.csv"

_PANDAS_NA = {"nan", "NaN", "NA", "N/A", "n/a", "null", "NULL", "", "#N/A", "#NA"}

n_before = len(df)
n_empty_tokens = (df["n_tokens_proc"] == 0).sum()
n_nan_string = df[df["n_tokens_proc"] > 0]["review_processed"].isin(_PANDAS_NA).sum()
print(f"Tokens vazios:       {n_empty_tokens}")
print(f"String nan residual: {n_nan_string}")

df_final = df[
    (df["n_tokens_proc"] > 0) &
    (~df["review_processed"].isin(_PANDAS_NA))
].copy()
n_dropped = n_before - len(df_final)
print(f"Total removidas: {n_dropped} reviews ({n_dropped/n_before*100:.1f}%)")
print(f"Shape salvo: {df_final[cols_to_save].shape}")
print(df_final["sentiment"].value_counts())

df_final[cols_to_save].to_csv(output_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
print(f"\nSalvo: {output_path}")


Tokens vazios:       13
String nan residual: 1


Total removidas: 14 reviews (0.4%)
Shape salvo: (3356, 4)
sentiment
positivo    1636
negativo    1434
neutro       286
Name: count, dtype: int64

Salvo: ../data/processed/zomato_reviews_processed.csv


## 8. Conclusões e decisões para o nb03

### Achados

| Item | Valor |
|---|---|
| Shape de entrada | 3.370 reviews |
| Redução média de tokens por review | 12,9 → 8,1 (37,1% menos) |
| Vocabulário único | 4.058 tokens (era 4.188 antes das correções desta sessão, -130) |
| Reviews sem tokens após o pipeline | 13 (Hindi/Gujarati puro ou só stopwords) |
| String `'nan'` residual | 1 |
| **Shape final salvo** | **3.356 reviews (99,6% do dataset de entrada)** |
| Distribuição final | positivo 1.636 / negativo 1.434 / neutro 286 |
| `le` (artefato de "less" sem POS) | 158 → 19 reviews (queda de 88%) |
| `worst`/`worse` → `bad` (fusão WordNet) | `worst` cai pra 68 reviews residuais; `bad` sobe pra 196 (positivo) / 180 (negativo) / 54 (neutro) |
| `not`/`no`/`nor` (negação, antes inexistentes como token) | 910/228/7 reviews, sem discriminar sentimento isolados (p=0,234), candidatos a bigrama no nb03 |

### Decisões tomadas

- Pipeline reaproveitado de `src/preprocessing.py` (`clean_html`, `expand_contractions`, `tokenize_raw`, `tag_pos`, `remove_stopwords`, `lemmatize_tagged`)
- **Lematização passou a usar POS tagging real** (`nltk.pos_tag`), corrigindo o trade-off aceito nas sessões anteriores (lematização sem POS, assumindo substantivo por padrão). Motivação: investigação na sessão de revisão do nb03 encontrou 7 tokens de alta frequência afetados (`packing`, `packaging`, `received`, `got`, `delivered`, `ordered`, `given`)
- H1 confirma que a limpeza não distorce nenhuma classe desproporcionalmente, a decisão de aplicar o mesmo pipeline às três classes está respaldada por teste estatístico
- H2-H3 confirmam que `test` e `tha`, tokens de alta frequência com forma estranha no top-20/ranking curto por classe, não são ruído do pipeline: `test` é grafia Hinglish de "taste" (99,4% de exclusão mútua com o token "taste" na mesma review), `tha` é palavra funcional hindi (88 reviews, sempre em contexto Hinglish). Nenhum exige stopwords customizado nesta fase
- H4, ao contrário de H2-H3, confirma que `amp` **é** ruído do pipeline: resíduo da entidade HTML `&amp;` (68 reviews no texto bruto) não decodificada por `clean_html` antes da remoção de caracteres não alfabéticos. Corrigido com `html.unescape()` em `src/preprocessing.py`, removeu 92 ocorrências do token
- **Achado lateral `worst`/`worse` → `bad`:** com POS correto, o `WordNetLemmatizer` trata `worst`/`worse` como formas irregulares de `bad` (comportamento documentado do dicionário WordNet, não bug). Aceito sem lista de exceção, mesmo princípio da correção geral (concentrar sinal fragmentado). 68 reviews mantêm `worst` intacto (maiúscula no início confundindo o tagger, ou uso adverbial genuíno)
- **Correção de contrações `n't`:** a troca de tokenizador (necessária pro POS tagging) gerava fragmentos sem sentido a partir de contrações. `expand_contractions()` resolve isso na origem, e `not`/`no`/`nor` saíram da lista de stopwords pra preservar negação como sinal, já capturável em bigramas no nb03 (`not bad`, `not good`)

### Próximo passo, notebook 03

- Reconstrução  do nb03 sobre o novo `data/processed/zomato_reviews_processed.csv` (vocabulário mudou)
- Reavaliar se as stopwords customizadas `packing`/`packaging`/`also` do grid `EXPERIMENTS` ainda fazem sentido dado o vocabulário novo
- Testar `not`/`no` combinados com bigramas (`ngram_range=(1,2)`) pra capturar negação (`not bad`, `not good`)
- Vetorização TF-IDF (Format Data)
- Seleção explícita de features além de `min_df` (Select Data, nível coluna)
- Split de validação separado do teste na comparação de configurações/modelos